# MolForge GRPO RL Training

This notebook runs a short GRPO training pass from the Qwen3.5 2B SFT adapter against the MolForge OpenEnv reward. It saves trainer logs, completion-level reward diagnostics, before/after evaluation JSON, plots, adapter checkpoints, and a zip archive to Google Drive.

In [ ]:
!nvidia-smi
!pip install -U "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -U "trl>=0.21.0" peft accelerate bitsandbytes datasets matplotlib pandas huggingface_hub

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!rm -rf /content/molt_lab
!git clone https://github.com/Adhitya-Vardhan/molt_lab.git /content/molt_lab
%cd /content/molt_lab

In [ ]:
# Pull the current v4 adapter from the Hugging Face Space unless you already have it in Drive.
!hf download Adhitya122/molt-lab --repo-type space --include 'adapters/qwen3_5_2b_lora_adapters_compact_v4/*' --local-dir /content/molforge_space

import os
os.environ['SFT_ADAPTER_PATH'] = '/content/molforge_space/adapters/qwen3_5_2b_lora_adapters_compact_v4'
os.environ['DRIVE_OUTPUT_DIR'] = '/content/drive/MyDrive/MolForge_RL_Runs'
os.environ['MAX_SEQ_LENGTH'] = '2048'
os.environ['MAX_PROMPT_LENGTH'] = '1536'
os.environ['MAX_COMPLETION_LENGTH'] = '384'
os.environ['RL_MAX_STEPS'] = '80'
os.environ['NUM_GENERATIONS'] = '2'
os.environ['RL_PROMPT_EPISODES'] = '96'
os.environ['RL_BATCH_SIZE'] = '2'
os.environ['RL_GRAD_ACCUM'] = '4'
os.environ['RL_LEARNING_RATE'] = '2e-6'
os.environ['RUN_EVAL'] = '1'

print('Adapter:', os.environ['SFT_ADAPTER_PATH'])
print('Drive output:', os.environ['DRIVE_OUTPUT_DIR'])

In [ ]:
!python issue/qwen3_5_2b_unsloth_grpo_rl.py

In [ ]:
import json, os
from pathlib import Path
from IPython.display import Image, display

run_root = Path('/content/molforge_rl_runs')
latest = max(run_root.iterdir(), key=lambda p: p.stat().st_mtime)
print('Latest run:', latest)
print((latest / 'rl_summary.json').read_text()[:2000])
for name in ['reward_curve.png', 'loss_curve.png', 'eval_before_after.png', 'action_distribution.png']:
    path = latest / 'plots' / name
    if path.exists():
        print(path)
        display(Image(filename=str(path)))